In [1]:
import pandas as pd
import re as re
from pathlib import Path

In [2]:
RUTA_PROYECTO = Path("..")
RUTA_INTERIM = RUTA_PROYECTO / "data" / "interim"
RUTA_PROCESSED = RUTA_PROYECTO / "data" / "processed"

carpeta_resenas = (
    RUTA_INTERIM / "resenas_videojuegos_bloques"
)

archivos_resenas = sorted(
    carpeta_resenas.glob("*.parquet")
)

RUTA_PROCESSED.mkdir(
    parents=True,
    exist_ok=True
)

carpeta_salida = RUTA_PROCESSED / "resenas_sin_duplicados"

# Limpiar la carpeta de salida si ya contiene archivos generados

if carpeta_salida.exists():
    archivos_anteriores = list(
        carpeta_salida.glob("*.parquet")
    )

    if archivos_anteriores:
        print(
            f"Se han encontrado {len(archivos_anteriores)} "
            "archivos anteriores."
        )

        for archivo in archivos_anteriores:
            archivo.unlink()

        print("Archivos anteriores eliminados correctamente.")

    else:
        print("La carpeta existe, pero no contiene archivos Parquet.")

else:
    carpeta_salida.mkdir(
        parents=True,
        exist_ok=True
    )

    print("La carpeta de salida no existía y ha sido creada.")


print("COMPROBACIÓN DE ARCHIVOS")
print("------------------------")
print(
    "Archivos de reseñas encontrados:",
    len(archivos_resenas)
)
print(
    "Carpeta de salida:",
    carpeta_salida.resolve()
)

Se han encontrado 47 archivos anteriores.
Archivos anteriores eliminados correctamente.
COMPROBACIÓN DE ARCHIVOS
------------------------
Archivos de reseñas encontrados: 47
Carpeta de salida: C:\Users\Paula María\Desktop\IS\5º CURSO\TFG\Trabajo-Fin-de-Grado\data\processed\resenas_sin_duplicados


In [3]:
# Selección de columnas relevantes
columnas_necesarias = [
    "rating",
    "title",
    "text",
    "parent_asin",
    "user_id",
    "timestamp",
    "helpful_vote",
    "verified_purchase"
]


In [4]:
# Comprobación
primer_bloque = pd.read_parquet(
    archivos_resenas[0]
)

columnas_disponibles = [
    columna
    for columna in columnas_necesarias
    if columna in primer_bloque.columns
]

columnas_no_disponibles = [
    columna
    for columna in columnas_necesarias
    if columna not in primer_bloque.columns
]

print("COLUMNAS SELECCIONADAS")
print("----------------------")

for columna in columnas_disponibles:
    print("-", columna)

print("\nColumnas no encontradas:")
print(columnas_no_disponibles)

COLUMNAS SELECCIONADAS
----------------------
- rating
- title
- text
- parent_asin
- user_id
- timestamp
- helpful_vote
- verified_purchase

Columnas no encontradas:
[]


In [5]:
columnas_duplicados = [
    "user_id",
    "parent_asin",
    "title",
    "text",
    "timestamp"
]

huellas_por_bloque = []

for archivo in archivos_resenas:
    bloque = pd.read_parquet(
        archivo,
        columns=columnas_duplicados
    )

    huellas = pd.util.hash_pandas_object(
        bloque,
        index=False
    )

    huellas_por_bloque.append(huellas)

In [6]:
todas_las_huellas = pd.concat(
    huellas_por_bloque,
    ignore_index=True
)

numero_duplicados = (
    todas_las_huellas
    .duplicated()
    .sum()
)

porcentaje_duplicados = (
    numero_duplicados
    / len(todas_las_huellas)
    * 100
)

print("ANÁLISIS DE DUPLICADOS")
print("----------------------")
print(
    "Reseñas analizadas:",
    f"{len(todas_las_huellas):,}"
)
print(
    "Posibles duplicados:",
    f"{numero_duplicados:,}"
)
print(
    "Porcentaje:",
    f"{porcentaje_duplicados:.4f} %"
)

ANÁLISIS DE DUPLICADOS
----------------------
Reseñas analizadas: 1,502,718
Posibles duplicados: 17,330
Porcentaje: 1.1532 %


In [7]:
# Identificación de reseñas duplicadas
huellas_duplicadas = set(
    todas_las_huellas[
        todas_las_huellas.duplicated(keep=False)
    ]
)

print(
    "Huellas correspondientes a grupos duplicados:",
    f"{len(huellas_duplicadas):,}"
)

Huellas correspondientes a grupos duplicados: 14,623


In [8]:
muestras_duplicadas = []

for archivo in archivos_resenas:
    bloque = pd.read_parquet(
        archivo,
        columns=columnas_duplicados
    )

    bloque = bloque.copy()

    bloque["huella"] = pd.util.hash_pandas_object(
        bloque[columnas_duplicados],
        index=False
    )

    coincidencias = bloque[
        bloque["huella"].isin(huellas_duplicadas)
    ]

    if not coincidencias.empty:
        muestras_duplicadas.append(coincidencias)


In [9]:
resenas_duplicadas = pd.concat(
    muestras_duplicadas,
    ignore_index=True
)

resenas_duplicadas = resenas_duplicadas.sort_values(
    by="huella"
)

print(
    "Filas pertenecientes a grupos duplicados:",
    f"{len(resenas_duplicadas):,}"
)

resenas_duplicadas.head(20)

Filas pertenecientes a grupos duplicados: 31,953


,user_id,parent_asin,title,text,timestamp,huella
18096,AH5F467CVWIKNE734F7G4CMM3RWA,B07YBXFDYN,Holy crap I love this game!,This is the best game that I've played in my e...,2015-07-05 02:52:10.000,285955783641270
18098,AH5F467CVWIKNE734F7G4CMM3RWA,B07YBXFDYN,Holy crap I love this game!,This is the best game that I've played in my e...,2015-07-05 02:52:10.000,285955783641270
18097,AH5F467CVWIKNE734F7G4CMM3RWA,B07YBXFDYN,Holy crap I love this game!,This is the best game that I've played in my e...,2015-07-05 02:52:10.000,285955783641270
18095,AH5F467CVWIKNE734F7G4CMM3RWA,B07YBXFDYN,Holy crap I love this game!,This is the best game that I've played in my e...,2015-07-05 02:52:10.000,285955783641270
6451,AFO5DWYZQM6ZRGJPKE5ECVNX635A,B07X2H4KDG,Five Stars,Great game for my husband and son,2016-03-15 11:58:32.000,1575817210768447
6453,AFO5DWYZQM6ZRGJPKE5ECVNX635A,B07X2H4KDG,Five Stars,Great game for my husband and son,2016-03-15 11:58:32.000,1575817210768447
8279,AH5T7VO635CDNXNPJEUBP63OGDHQ,B00008URUA,one of the greatest video games ever!,This game is awesome! I love it!,2004-01-17 19:40:55.000,2810222883300917
8277,AH5T7VO635CDNXNPJEUBP63OGDHQ,B00008URUA,one of the greatest video games ever!,This game is awesome! I love it!,2004-01-17 19:40:55.000,2810222883300917
27956,AG43E6EBXSSSEWBXCPFSNAVB4F6A,B001EYUU1U,as described,as described,2023-01-09 19:15:47.195,3088448328175193
27941,AG43E6EBXSSSEWBXCPFSNAVB4F6A,B001EYUU1U,as described,as described,2023-01-09 19:15:47.195,3088448328175193


In [10]:
resenas_duplicadas[
    [
        "user_id",
        "parent_asin",
        "title",
        "text",
        "timestamp"
    ]
].head(20)

,user_id,parent_asin,title,text,timestamp
18096,AH5F467CVWIKNE734F7G4CMM3RWA,B07YBXFDYN,Holy crap I love this game!,This is the best game that I've played in my e...,2015-07-05 02:52:10.000
18098,AH5F467CVWIKNE734F7G4CMM3RWA,B07YBXFDYN,Holy crap I love this game!,This is the best game that I've played in my e...,2015-07-05 02:52:10.000
18097,AH5F467CVWIKNE734F7G4CMM3RWA,B07YBXFDYN,Holy crap I love this game!,This is the best game that I've played in my e...,2015-07-05 02:52:10.000
18095,AH5F467CVWIKNE734F7G4CMM3RWA,B07YBXFDYN,Holy crap I love this game!,This is the best game that I've played in my e...,2015-07-05 02:52:10.000
6451,AFO5DWYZQM6ZRGJPKE5ECVNX635A,B07X2H4KDG,Five Stars,Great game for my husband and son,2016-03-15 11:58:32.000
6453,AFO5DWYZQM6ZRGJPKE5ECVNX635A,B07X2H4KDG,Five Stars,Great game for my husband and son,2016-03-15 11:58:32.000
8279,AH5T7VO635CDNXNPJEUBP63OGDHQ,B00008URUA,one of the greatest video games ever!,This game is awesome! I love it!,2004-01-17 19:40:55.000
8277,AH5T7VO635CDNXNPJEUBP63OGDHQ,B00008URUA,one of the greatest video games ever!,This game is awesome! I love it!,2004-01-17 19:40:55.000
27956,AG43E6EBXSSSEWBXCPFSNAVB4F6A,B001EYUU1U,as described,as described,2023-01-09 19:15:47.195
27941,AG43E6EBXSSSEWBXCPFSNAVB4F6A,B001EYUU1U,as described,as described,2023-01-09 19:15:47.195


In [11]:
frecuencia_huellas = (
    todas_las_huellas
    .value_counts()
)

grupos_repetidos = frecuencia_huellas[
    frecuencia_huellas > 1
]

print("RESUMEN DE GRUPOS DUPLICADOS")
print("----------------------------")
print(
    "Grupos de reseñas repetidas:",
    f"{len(grupos_repetidos):,}"
)
print(
    "Máximo número de apariciones:",
    f"{grupos_repetidos.max():,}"
)


RESUMEN DE GRUPOS DUPLICADOS
----------------------------
Grupos de reseñas repetidas: 14,623
Máximo número de apariciones: 10


In [12]:
#Eliminación de duplicados

huellas_encontradas = set() #Huellas que ya han aparecido

total_reseñas_originales = 0 # Todas las reseñas antes de limpieza
total_reseñas_conservadas = 0 # Todas las reseñas después de limpieza
total_duplicados_eliminados = 0 # Todas las reseñas eliminadas por duplicados

In [13]:
for numero_bloque, archivo in enumerate(
    archivos_resenas,
    start=1
):
    # Cargar el bloque completo
    bloque = pd.read_parquet(archivo)

    total_reseñas_originales += len(bloque)

    # Generar la huella de cada reseña
    huellas_bloque = pd.util.hash_pandas_object(
        bloque[columnas_duplicados],
        index=False
    )

    # Detectar reseñas que ya aparecieron en archivos anteriores
    duplicadas_anteriores = huellas_bloque.isin(
        huellas_encontradas
    )

    # Detectar repeticiones dentro del propio bloque
    duplicadas_bloque = huellas_bloque.duplicated(
        keep="first"
    )

    # Una reseña es duplicada si cumple cualquiera de las condiciones
    mascara_duplicados = (
        duplicadas_anteriores
        | duplicadas_bloque
    )

    # Conservar únicamente las filas no duplicadas
    bloque_limpio = bloque[
        ~mascara_duplicados
    ].copy()

    # Añadir al conjunto las huellas que se han conservado
    huellas_conservadas = huellas_bloque[
        ~mascara_duplicados
    ]

    huellas_encontradas.update(
        huellas_conservadas.tolist()
    )

    # Actualizar los contadores
    duplicados_bloque = int(
        mascara_duplicados.sum()
    )

    total_duplicados_eliminados += duplicados_bloque
    total_reseñas_conservadas += len(bloque_limpio)

    # Guardar el bloque limpio
    ruta_archivo_salida = carpeta_salida / (
        f"resenas_sin_duplicados_{numero_bloque - 1:04d}.parquet"
    )

    bloque_limpio.to_parquet(
        ruta_archivo_salida,
        index=False
    )

    print(
        f"Bloque {numero_bloque}/{len(archivos_resenas)}: "
        f"{len(bloque):,} revisadas, "
        f"{duplicados_bloque:,} eliminadas, "
        f"{len(bloque_limpio):,} conservadas"
    )

Bloque 1/47: 39,231 revisadas, 56 eliminadas, 39,175 conservadas
Bloque 2/47: 32,353 revisadas, 161 eliminadas, 32,192 conservadas
Bloque 3/47: 33,745 revisadas, 81 eliminadas, 33,664 conservadas
Bloque 4/47: 35,227 revisadas, 104 eliminadas, 35,123 conservadas
Bloque 5/47: 33,000 revisadas, 124 eliminadas, 32,876 conservadas
Bloque 6/47: 34,064 revisadas, 137 eliminadas, 33,927 conservadas
Bloque 7/47: 35,931 revisadas, 145 eliminadas, 35,786 conservadas
Bloque 8/47: 41,028 revisadas, 236 eliminadas, 40,792 conservadas
Bloque 9/47: 33,160 revisadas, 242 eliminadas, 32,918 conservadas
Bloque 10/47: 34,812 revisadas, 255 eliminadas, 34,557 conservadas
Bloque 11/47: 37,489 revisadas, 213 eliminadas, 37,276 conservadas
Bloque 12/47: 33,658 revisadas, 174 eliminadas, 33,484 conservadas
Bloque 13/47: 36,308 revisadas, 128 eliminadas, 36,180 conservadas
Bloque 14/47: 33,566 revisadas, 166 eliminadas, 33,400 conservadas
Bloque 15/47: 29,105 revisadas, 129 eliminadas, 28,976 conservadas
Bloque

In [14]:
print("\nRESULTADO DE LA ELIMINACIÓN")
print("---------------------------")
print(
    "Reseñas antes de la limpieza:",
    f"{total_reseñas_originales:,}"
)
print(
    "Copias duplicadas eliminadas:",
    f"{total_duplicados_eliminados:,}"
)
print(
    "Reseñas después de la limpieza:",
    f"{total_reseñas_conservadas:,}"
)


RESULTADO DE LA ELIMINACIÓN
---------------------------
Reseñas antes de la limpieza: 1,502,718
Copias duplicadas eliminadas: 17,330
Reseñas después de la limpieza: 1,485,388


In [15]:
print(
    "¿El resultado es coherente?",
    total_reseñas_originales
    - total_duplicados_eliminados
    == total_reseñas_conservadas
)

¿El resultado es coherente? True


In [16]:
# Comprobación de archivos de salida
archivos_sin_duplicados = sorted(
    carpeta_salida.glob("*.parquet")
)

filas_guardadas = 0

for archivo in archivos_sin_duplicados:
    bloque = pd.read_parquet(
        archivo,
        columns=["parent_asin"]
    )

    filas_guardadas += len(bloque)

print("COMPROBACIÓN DE LOS ARCHIVOS")
print("----------------------------")
print(
    "Archivos generados:",
    len(archivos_sin_duplicados)
)
print(
    "Filas realmente guardadas:",
    f"{filas_guardadas:,}"
)
print(
    "¿Coinciden con el contador?",
    filas_guardadas == total_reseñas_conservadas
)

COMPROBACIÓN DE LOS ARCHIVOS
----------------------------
Archivos generados: 47
Filas realmente guardadas: 1,485,388
¿Coinciden con el contador? True


In [17]:
# Verificación final

huellas_limpias_por_bloque = []

for archivo in archivos_sin_duplicados:
    bloque = pd.read_parquet(
        archivo,
        columns=columnas_duplicados
    )

    huellas = pd.util.hash_pandas_object(
        bloque,
        index=False
    )

    huellas_limpias_por_bloque.append(
        huellas
    )
todas_las_huellas_limpias = pd.concat(
    huellas_limpias_por_bloque,
    ignore_index=True
)

duplicados_restantes = (
    todas_las_huellas_limpias
    .duplicated()
    .sum()
)

print("\nCOMPROBACIÓN FINAL")
print("------------------")
print(
    "Reseñas analizadas:",
    f"{len(todas_las_huellas_limpias):,}"
)
print(
    "Duplicados restantes:",
    f"{duplicados_restantes:,}"
)


COMPROBACIÓN FINAL
------------------
Reseñas analizadas: 1,485,388
Duplicados restantes: 0


In [18]:
# Comprobación archivos vacios

textos_vacios = 0
titulos_vacios = 0

for archivo in archivos_sin_duplicados:
    bloque = pd.read_parquet(
        archivo,
        columns=["title", "text"]
    )

    textos_vacios += (
        bloque["text"]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    titulos_vacios += (
        bloque["title"]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

print("COMPROBACIÓN DE TEXTOS VACÍOS")
print("-----------------------------")
print(f"Textos vacíos: {textos_vacios:,}")
print(f"Títulos vacíos: {titulos_vacios:,}")

COMPROBACIÓN DE TEXTOS VACÍOS
-----------------------------
Textos vacíos: 398
Títulos vacíos: 0


In [19]:
muestras_texto_vacio = []

for archivo in archivos_sin_duplicados:
    bloque = pd.read_parquet(
        archivo,
        columns=[
            "rating",
            "title",
            "text",
            "parent_asin"
        ]
    )

    mascara_texto_vacio = (
        bloque["text"]
        .astype(str)
        .str.strip()
        .eq("")
    )

    seleccion = bloque[
        mascara_texto_vacio
    ].copy()

    if not seleccion.empty:
        muestras_texto_vacio.append(seleccion)

In [20]:
# Comprobación de reseñas con texto vacío
resenas_texto_vacio = pd.concat(
    muestras_texto_vacio,
    ignore_index=True
)

print(
    "Reseñas con texto vacío:",
    f"{len(resenas_texto_vacio):,}"
)

resenas_texto_vacio.head(20)

Reseñas con texto vacío: 398


,rating,title,text,parent_asin
0,4,Bien!,,B00C1TTF86
1,5,Five Stars,,B01AC0I9NW
2,5,Great Game,,B08GR4SWCM
3,5,AMAZING❤️,,B087NNZZM8
4,5,There are two seperate games in a sleve.,,B06XXX3R11
5,5,Five Stars,,B01CHU4IY4
6,5,Five Stars,,B00Z9TLVK0
7,5,Osum!,,B0050QOCMQ
8,4,Just in time!,,B008MWIALE
9,5,¡She loves it!,,B00HBUOVIO


In [21]:
# Comprobación de reseñas con título y texto vacíos
ambos_campos_vacios = 0

for archivo in archivos_sin_duplicados:
    bloque = pd.read_parquet(
        archivo,
        columns=["title", "text"]
    )

    titulo_vacio = (
        bloque["title"]
        .astype(str)
        .str.strip()
        .eq("")
    )

    texto_vacio = (
        bloque["text"]
        .astype(str)
        .str.strip()
        .eq("")
    )

    ambos_campos_vacios += (
        titulo_vacio & texto_vacio
    ).sum()

print(
    "Reseñas con título y texto vacíos:",
    f"{ambos_campos_vacios:,}"
)

Reseñas con título y texto vacíos: 0


In [22]:
# Como existen reseñas con el campo text vacío, pero todas disponen de título,
# se combinan title y text en una nueva columna llamada review_text.
# De este modo, se conserva la información del título y no se elimina ninguna reseña.

bloque_prueba = pd.read_parquet(
    archivos_sin_duplicados[0])

titulo_limpio= ( bloque_prueba["title"]
    .astype(str)
    .str.strip())
texto_limpio = (
    bloque_prueba["text"]
    .astype(str)
    .str.strip()
)

bloque_prueba["review_text"] = (
    titulo_limpio +" "+ texto_limpio
    ).str.strip()

In [23]:
bloque_prueba[["title", "text", "review_text", "rating"]].head(10)

,title,text,review_text,rating
0,It’s pretty sexual. Not my fav,I’m playing on ps5 and it’s interesting. It’s...,It’s pretty sexual. Not my fav I’m playing on ...,4
1,Good. A bit slow,Nostalgic fun. A bit slow. I hope they don’t...,Good. A bit slow Nostalgic fun. A bit slow. ...,5
2,... an order for my kids & they have really en...,This was an order for my kids & they have real...,... an order for my kids & they have really en...,5
3,What a great game! Fun for everyone and keeps ...,"So, I got this and used this game for the firs...",What a great game! Fun for everyone and keeps ...,5
4,Disney Lion King 1.5,Updating my Disney movies to Blu Ray. was happ...,Disney Lion King 1.5 Updating my Disney movies...,4
5,LOVE THE UDRAW,I bought this as a Christmas present to my gra...,LOVE THE UDRAW I bought this as a Christmas pr...,5
6,Good Chocolates,Husband bought these for me for our 25th Anniv...,Good Chocolates Husband bought these for me fo...,5
7,Monopoly variety game for Switch,I purchased a Switch to be able to play with m...,Monopoly variety game for Switch I purchased a...,3
8,Non stop fun,I still own my Wii which works perfectly and I...,Non stop fun I still own my Wii which works pe...,5
9,did not work for more than an hour,Stopped working an hour after my son started p...,did not work for more than an hour Stopped wor...,1


In [24]:
#Comprobación de que ningun nuevo texto esté vacío

textos_combinados_vacios = (
    bloque_prueba["review_text"]
    .str.strip()
    .eq("")
    .sum()
)

print(
    "Textos combinados vacíos:",
    textos_combinados_vacios
)

Textos combinados vacíos: 0


In [25]:
# Limpiar la carpeta de salida si ya contiene archivos generados

carpeta_texto_combinado = (
    RUTA_PROCESSED / "resenas_texto_combinado"
)


if carpeta_texto_combinado.exists():
    archivos_anteriores = list(
        carpeta_texto_combinado.glob("*.parquet")
    )

    if archivos_anteriores:
        print(
            f"Se han encontrado {len(archivos_anteriores)} "
            "archivos anteriores."
        )

        for archivo in archivos_anteriores:
            archivo.unlink()

        print("Archivos anteriores eliminados correctamente.")

    else:
        print("La carpeta existe, pero no contiene archivos Parquet.")

else:
    carpeta_texto_combinado.mkdir(
        parents=True,
        exist_ok=True
    )

    print("La carpeta de salida no existía y ha sido creada.")


print(
    "Carpeta de salida:",
    carpeta_texto_combinado.resolve()
)

Se han encontrado 47 archivos anteriores.
Archivos anteriores eliminados correctamente.
Carpeta de salida: C:\Users\Paula María\Desktop\IS\5º CURSO\TFG\Trabajo-Fin-de-Grado\data\processed\resenas_texto_combinado


In [26]:
archivos_anteriores = sorted(
    carpeta_texto_combinado.glob("*.parquet")
)

print(
    "Archivos anteriores:",
    len(archivos_anteriores)
)


Archivos anteriores: 0


In [27]:
total_reseñas_procesadas = 0
total_review_text_vacios = 0
total_cuerpos_vacios = 0

for numero_bloque, archivo in enumerate(
    archivos_sin_duplicados,
    start=1
):
    # Cargar el bloque
    bloque = pd.read_parquet(archivo)

    # Preparar los títulos
    titulo_limpio = (
        bloque["title"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    # Preparar los cuerpos de las reseñas
    texto_limpio = (
        bloque["text"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    # Contar los cuerpos vacíos
    cuerpos_vacios_bloque = (
        texto_limpio.eq("").sum()
    )

    total_cuerpos_vacios += cuerpos_vacios_bloque

    # Crear la nueva columna combinando título y texto
    bloque["review_text"] = (
        titulo_limpio + " " + texto_limpio
    ).str.strip()

    # Comprobar si algún texto combinado queda vacío
    review_text_vacios_bloque = (
        bloque["review_text"]
        .eq("")
        .sum()
    )

    total_review_text_vacios += (
        review_text_vacios_bloque
    )

    total_reseñas_procesadas += len(bloque)

    # Construir el nombre del archivo de salida
    ruta_salida = carpeta_texto_combinado / (
        f"resenas_texto_combinado_"
        f"{numero_bloque - 1:04d}.parquet"
    )

    # Guardar el bloque con la nueva columna
    bloque.to_parquet(
        ruta_salida,
        index=False
    )

    print(
        f"Bloque {numero_bloque}/{len(archivos_sin_duplicados)}: "
        f"{len(bloque):,} reseñas procesadas, "
        f"{cuerpos_vacios_bloque:,} cuerpos vacíos, "
        f"{review_text_vacios_bloque:,} textos combinados vacíos"
    )

Bloque 1/47: 39,175 reseñas procesadas, 0 cuerpos vacíos, 0 textos combinados vacíos
Bloque 2/47: 32,192 reseñas procesadas, 3 cuerpos vacíos, 0 textos combinados vacíos
Bloque 3/47: 33,664 reseñas procesadas, 4 cuerpos vacíos, 0 textos combinados vacíos
Bloque 4/47: 35,123 reseñas procesadas, 0 cuerpos vacíos, 0 textos combinados vacíos
Bloque 5/47: 32,876 reseñas procesadas, 4 cuerpos vacíos, 0 textos combinados vacíos
Bloque 6/47: 33,927 reseñas procesadas, 1 cuerpos vacíos, 0 textos combinados vacíos
Bloque 7/47: 35,786 reseñas procesadas, 4 cuerpos vacíos, 0 textos combinados vacíos
Bloque 8/47: 40,792 reseñas procesadas, 3 cuerpos vacíos, 0 textos combinados vacíos
Bloque 9/47: 32,918 reseñas procesadas, 5 cuerpos vacíos, 0 textos combinados vacíos
Bloque 10/47: 34,557 reseñas procesadas, 4 cuerpos vacíos, 0 textos combinados vacíos
Bloque 11/47: 37,276 reseñas procesadas, 3 cuerpos vacíos, 0 textos combinados vacíos
Bloque 12/47: 33,484 reseñas procesadas, 4 cuerpos vacíos, 0 te

In [28]:
print("\nRESULTADO DE LA COMBINACIÓN")
print("---------------------------")
print(
    "Reseñas procesadas:",
    f"{total_reseñas_procesadas:,}"
)
print(
    "Reseñas con cuerpo vacío:",
    f"{total_cuerpos_vacios:,}"
)
print(
    "Textos combinados vacíos:",
    f"{total_review_text_vacios:,}"
)


RESULTADO DE LA COMBINACIÓN
---------------------------
Reseñas procesadas: 1,485,388
Reseñas con cuerpo vacío: 398
Textos combinados vacíos: 0


In [29]:
# Comprobación de archivos de salida
archivos_texto_combinado = sorted(
    carpeta_texto_combinado.glob("*.parquet")
)

filas_guardadas = 0

for archivo in archivos_texto_combinado:
    bloque = pd.read_parquet(
        archivo,
        columns=["review_text"]
    )

    filas_guardadas += len(bloque)

print("\nCOMPROBACIÓN DE ARCHIVOS")
print("------------------------")
print(
    "Archivos generados:",
    len(archivos_texto_combinado)
)
print(
    "Filas guardadas:",
    f"{filas_guardadas:,}"
)
print(
    "¿Coinciden con las reseñas procesadas?",
    filas_guardadas == total_reseñas_procesadas
)


COMPROBACIÓN DE ARCHIVOS
------------------------
Archivos generados: 47
Filas guardadas: 1,485,388
¿Coinciden con las reseñas procesadas? True


In [30]:
# Muestra de los archivos generados

bloque_resultado = pd.read_parquet(
    archivos_texto_combinado[0]
)

bloque_resultado[
    [
        "title",
        "text",
        "review_text",
        "rating"
    ]
].head(10)

,title,text,review_text,rating
0,It’s pretty sexual. Not my fav,I’m playing on ps5 and it’s interesting. It’s...,It’s pretty sexual. Not my fav I’m playing on ...,4
1,Good. A bit slow,Nostalgic fun. A bit slow. I hope they don’t...,Good. A bit slow Nostalgic fun. A bit slow. ...,5
2,... an order for my kids & they have really en...,This was an order for my kids & they have real...,... an order for my kids & they have really en...,5
3,What a great game! Fun for everyone and keeps ...,"So, I got this and used this game for the firs...",What a great game! Fun for everyone and keeps ...,5
4,Disney Lion King 1.5,Updating my Disney movies to Blu Ray. was happ...,Disney Lion King 1.5 Updating my Disney movies...,4
5,LOVE THE UDRAW,I bought this as a Christmas present to my gra...,LOVE THE UDRAW I bought this as a Christmas pr...,5
6,Good Chocolates,Husband bought these for me for our 25th Anniv...,Good Chocolates Husband bought these for me fo...,5
7,Monopoly variety game for Switch,I purchased a Switch to be able to play with m...,Monopoly variety game for Switch I purchased a...,3
8,Non stop fun,I still own my Wii which works perfectly and I...,Non stop fun I still own my Wii which works pe...,5
9,did not work for more than an hour,Stopped working an hour after my son started p...,did not work for more than an hour Stopped wor...,1


In [31]:
# NORMALIZACIÓN DEL TEXTO

import re

In [32]:
def normalizar_texto(texto):
    """
    Aplica una normalización inicial al texto de una reseña.

    Operaciones:
    1. Convierte el valor a texto.
    2. Convierte las letras a minúsculas.
    3. Elimina etiquetas HTML.
    4. Sustituye las URL por el marcador 'url'.
    5. Normaliza espacios, tabulaciones y saltos de línea.
    """

    # Convertir el valor a texto
    texto = str(texto)

    # Convertir a minúsculas
    texto = texto.lower()

    # Eliminar etiquetas HTML
    texto = re.sub(
        r"<[^>]+>",
        " ",
        texto
    )

    # Sustituir las URL por el marcador "url"
    texto = re.sub(
        r"https?://\S+|www\.\S+",
        " url ",
        texto
    )

    # Sustituir espacios, tabulaciones y saltos de línea
    # consecutivos por un único espacio
    texto = re.sub(
        r"\s+",
        " ",
        texto
    )

    # Eliminar espacios al principio y al final
    texto = texto.strip()

    return texto

In [33]:
textos_prueba = [
    """
    This GAME is <b>AMAZING</b>!
    More info: https://example.com/game
    """,
    "I do NOT like this game. It isn't good!",
    "Great     game!\nHighly recommended.",
    "<p>Very disappointing product.</p>",
    "Visit www.example.com for more information."
]

for numero, texto in enumerate(textos_prueba, start=1):
    print(f"PRUEBA {numero}")
    print("Original:")
    print(texto)

    print("Normalizado:")
    print(normalizar_texto(texto))
    print("-" * 50)


PRUEBA 1
Original:

    This GAME is <b>AMAZING</b>!
    More info: https://example.com/game
    
Normalizado:
this game is amazing ! more info: url
--------------------------------------------------
PRUEBA 2
Original:
I do NOT like this game. It isn't good!
Normalizado:
i do not like this game. it isn't good!
--------------------------------------------------
PRUEBA 3
Original:
Great     game!
Highly recommended.
Normalizado:
great game! highly recommended.
--------------------------------------------------
PRUEBA 4
Original:
<p>Very disappointing product.</p>
Normalizado:
very disappointing product.
--------------------------------------------------
PRUEBA 5
Original:
Visit www.example.com for more information.
Normalizado:
visit url for more information.
--------------------------------------------------


In [34]:
bloque_prueba["review_text_normalized"] = (
    bloque_prueba["review_text"]
    .apply(normalizar_texto)
)

In [35]:
bloque_prueba[
    [
        "review_text",
        "review_text_normalized"
    ]
].head(10)

,review_text,review_text_normalized
0,It’s pretty sexual. Not my fav I’m playing on ...,it’s pretty sexual. not my fav i’m playing on ...
1,Good. A bit slow Nostalgic fun. A bit slow. ...,good. a bit slow nostalgic fun. a bit slow. i ...
2,... an order for my kids & they have really en...,... an order for my kids & they have really en...
3,What a great game! Fun for everyone and keeps ...,what a great game! fun for everyone and keeps ...
4,Disney Lion King 1.5 Updating my Disney movies...,disney lion king 1.5 updating my disney movies...
5,LOVE THE UDRAW I bought this as a Christmas pr...,love the udraw i bought this as a christmas pr...
6,Good Chocolates Husband bought these for me fo...,good chocolates husband bought these for me fo...
7,Monopoly variety game for Switch I purchased a...,monopoly variety game for switch i purchased a...
8,Non stop fun I still own my Wii which works pe...,non stop fun i still own my wii which works pe...
9,did not work for more than an hour Stopped wor...,did not work for more than an hour stopped wor...


In [ ]:
# Comprobación de no textos vacíos tras la normalización

textos_normalizados_vacios = (
    bloque_prueba["review_text_normalized"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("COMPROBACIÓN DE LA NORMALIZACIÓN")
print("--------------------------------")
print(f"Reseñas analizadas: {len(bloque_prueba):,}")
print(
    "Textos normalizados vacíos:",
    f"{textos_normalizados_vacios:,}"
)



COMPROBACIÓN DE LA NORMALIZACIÓN
--------------------------------
Reseñas analizadas: 39,175
Textos normalizados vacíos: 0


In [52]:
total_resenas_procesadas = 0
total_textos_vacios = 0

ruta_normalizados = RUTA_PROCESSED / "resenas_normalizadas"

if not ruta_normalizados.exists():
    ruta_normalizados.mkdir(
        parents=True,
        exist_ok=True
    )

    print("La carpeta de salida no existía y ha sido creada.")

for numero_bloque, archivo in enumerate(
    archivos_texto_combinado,
    start=0
):
    # Cargar el bloque con la columna review_text
    bloque = pd.read_parquet(archivo)

    # Aplicar la normalización a cada reseña
    bloque["review_text_normalized"] = (
        bloque["review_text"]
        .apply(normalizar_texto)
    )

    # Contar los textos que hayan quedado vacíos
    textos_vacios_bloque = (
        bloque["review_text_normalized"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    total_textos_vacios += textos_vacios_bloque
    total_resenas_procesadas += len(bloque)

    # Construir el nombre del archivo de salida
    ruta_salida = ruta_normalizados / (
        f"resenas_normalizadas_"
        f"{numero_bloque:04d}.parquet"
    )

    # Guardar el bloque normalizado
    bloque.to_parquet(
        ruta_salida,
        index=False
    )

    print(
        f"Bloque {numero_bloque + 1}/"
        f"{len(archivos_texto_combinado)}: "
        f"{len(bloque):,} reseñas procesadas, "
        f"{textos_vacios_bloque:,} textos vacíos"
    )

Bloque 1/47: 39,175 reseñas procesadas, 0 textos vacíos
Bloque 2/47: 32,192 reseñas procesadas, 0 textos vacíos
Bloque 3/47: 33,664 reseñas procesadas, 0 textos vacíos
Bloque 4/47: 35,123 reseñas procesadas, 0 textos vacíos
Bloque 5/47: 32,876 reseñas procesadas, 0 textos vacíos
Bloque 6/47: 33,927 reseñas procesadas, 0 textos vacíos
Bloque 7/47: 35,786 reseñas procesadas, 0 textos vacíos
Bloque 8/47: 40,792 reseñas procesadas, 0 textos vacíos
Bloque 9/47: 32,918 reseñas procesadas, 0 textos vacíos
Bloque 10/47: 34,557 reseñas procesadas, 0 textos vacíos
Bloque 11/47: 37,276 reseñas procesadas, 0 textos vacíos
Bloque 12/47: 33,484 reseñas procesadas, 0 textos vacíos
Bloque 13/47: 36,180 reseñas procesadas, 0 textos vacíos
Bloque 14/47: 33,400 reseñas procesadas, 0 textos vacíos
Bloque 15/47: 28,976 reseñas procesadas, 0 textos vacíos
Bloque 16/47: 31,491 reseñas procesadas, 0 textos vacíos
Bloque 17/47: 31,264 reseñas procesadas, 0 textos vacíos
Bloque 18/47: 32,306 reseñas procesadas,

In [53]:
archivos_texto_combinado = sorted(
    carpeta_texto_combinado.glob("*.parquet")
)

print(
    "Archivos con texto combinado:",
    len(archivos_texto_combinado)
)


Archivos con texto combinado: 47


In [60]:
print("\nRESULTADO DE LA NORMALIZACIÓN")
print("-----------------------------")
print(
    "Reseñas procesadas:",
    f"{total_resenas_procesadas:,}"
)
print(
    "Textos normalizados vacíos:",
    f"{total_textos_vacios:,}"
)


RESULTADO DE LA NORMALIZACIÓN
-----------------------------
Reseñas procesadas: 1,485,388
Textos normalizados vacíos: 0


In [63]:
archivos_normalizados = sorted(
    ruta_normalizados.glob(
        "resenas_normalizadas_*.parquet"
    )
)

filas_normalizadas = 0

for archivo in archivos_texto_combinado:
    bloque = pd.read_parquet(
        archivo,
        columns=["review_text"]
    )

    filas_normalizadas += len(bloque)

print(
    "Reseñas de entrada:",
    f"{filas_normalizadas:,}"
)

filas_guardadas = 0
textos_vacios_verificados = 0

for archivo in archivos_normalizados:
    bloque = pd.read_parquet(
        archivo,
        columns=["review_text_normalized"]
    )

    filas_guardadas += len(bloque)

    textos_vacios_verificados += (
        bloque["review_text_normalized"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

print("\nVERIFICACIÓN FINAL")
print("------------------")
print(
    "Archivos normalizados:",
    len(archivos_normalizados)
)
print(
    "Filas realmente guardadas:",
    f"{filas_guardadas:,}"
)
print(
    "Textos normalizados vacíos:",
    f"{textos_vacios_verificados:,}"
)
print(
    "¿Se conserva el número de reseñas?",
    filas_guardadas == filas_normalizadas
)

Reseñas de entrada: 1,485,388

VERIFICACIÓN FINAL
------------------
Archivos normalizados: 47
Filas realmente guardadas: 1,485,388
Textos normalizados vacíos: 0
¿Se conserva el número de reseñas? True


resenas_normalizadas_0000.parquet
resenas_normalizadas_0001.parquet
resenas_normalizadas_0002.parquet
resenas_normalizadas_0003.parquet
resenas_normalizadas_0004.parquet
resenas_normalizadas_0005.parquet
resenas_normalizadas_0006.parquet
resenas_normalizadas_0007.parquet
resenas_normalizadas_0008.parquet
resenas_normalizadas_0009.parquet
resenas_normalizadas_0010.parquet
resenas_normalizadas_0011.parquet
resenas_normalizadas_0012.parquet
resenas_normalizadas_0013.parquet
resenas_normalizadas_0014.parquet
resenas_normalizadas_0015.parquet
resenas_normalizadas_0016.parquet
resenas_normalizadas_0017.parquet
resenas_normalizadas_0018.parquet
resenas_normalizadas_0019.parquet
resenas_normalizadas_0020.parquet
resenas_normalizadas_0021.parquet
resenas_normalizadas_0022.parquet
resenas_normalizadas_0023.parquet
resenas_normalizadas_0024.parquet
resenas_normalizadas_0025.parquet
resenas_normalizadas_0026.parquet
resenas_normalizadas_0027.parquet
resenas_normalizadas_0028.parquet
resenas_normal